In [133]:
import base64
from email.mime.text import MIMEText
from google.auth.transport.requests import Request
from google.oauth2.credentials import Credentials
from google_auth_oauthlib.flow import InstalledAppFlow
from googleapiclient.discovery import build

In [134]:
SCOPES = ["https://www.googleapis.com/auth/gmail.modify"]

In [135]:
def authenticate():
    creds = None
    if os.path.exists("token.json"):
        creds = Credentials.from_authorized_user_file("token.json", SCOPES)
    
    if not creds or not creds.valid:
        if creds and creds.expired and creds.refresh_token:
            creds.refresh(Request())
            print("Credentials refreshed.")
        else:
            flow = InstalledAppFlow.from_client_secrets_file("credentials.json", SCOPES)
            creds = flow.run_local_server(port=0)
            print("New credentials obtained.")

        with open("token.json", "w") as token:
            token.write(creds.to_json())
    else:
        print("Credentials loaded from token.json.")
    service = build("gmail", "v1", credentials=creds)
    return service


In [136]:
def get_message(service, gmail_id):
    msg_detail = service.users().messages().get(userId="me", id=gmail_id, format="full").execute()
    headers = msg_detail.get("payload", {}).get("headers", [])

    email_info = {
        "id": gmail_id,
        "threadId": msg_detail.get("threadId"),
        "labelIds": msg_detail.get("labelIds", []),
        "snippet": msg_detail.get("snippet"),
        "internalDate": msg_detail.get("internalDate"),
        "sizeEstimate": msg_detail.get("sizeEstimate"),
    }

    # Extract all headers into a dict
    for header in headers:
        name = header["name"].lower()
        if name in ("from", "to", "cc", "bcc", "subject", "date", "in-reply-to", "references", "message-id"):
            email_info[name] = header["value"]

    # Extract body
    payload = msg_detail.get("payload", {})
    parts = payload.get("parts", [])
    if parts:
        for part in parts:
            if part["mimeType"] == "text/plain":
                import base64
                email_info["body_text"] = base64.urlsafe_b64decode(part["body"]["data"]).decode("utf-8")
            elif part["mimeType"] == "text/html":
                email_info["body_html"] = base64.urlsafe_b64decode(part["body"]["data"]).decode("utf-8")
    elif payload.get("body", {}).get("data"):
        import base64
        email_info["body_text"] = base64.urlsafe_b64decode(payload["body"]["data"]).decode("utf-8")

    return email_info

In [137]:
def read_emails(service, max_results=10):
    results = service.users().messages().list(userId="me", maxResults=max_results).execute()
    messages = results.get("messages", [])
    email_data = []

    for msg in messages:
        email_data.append(get_message(service, msg["id"]))
    return email_data

In [138]:
def send_email(service, to, subject, body):
    message = MIMEText(body)
    message["to"] = to
    message["subject"] = subject

    raw = base64.urlsafe_b64encode(
        message.as_bytes()
    ).decode()

    sent = service.users().messages().send(
        userId="me", body={"raw": raw}
    ).execute()

    # Fetch the sent message to get its Message-ID
    try:
        sent_detail = service.users().messages().get(
            userId="me", id=sent["id"], format="metadata"
        ).execute()
        sent_headers = sent_detail["payload"]["headers"]
        msg_id = next(
            (h["value"] for h in sent_headers if h["name"].lower() == "message-id"),
            None
        )
    except Exception:
        msg_id = None

    return {
        "gmail_id": sent["id"],
        "thread_id": sent.get("threadId"),
        "message_id": msg_id,
    }

In [139]:
def reply_email(service, gmail_id, reply_text):
    original = service.users().messages().get(
        userId="me", id=gmail_id
    ).execute()

    headers = original["payload"]["headers"]
    subject = next(
        (h["value"] for h in headers if h["name"].lower() == "subject"), ""
    )
    from_email = next(
        (h["value"] for h in headers if h["name"].lower() == "from"), ""
    )
    original_msg_id = next(
        (h["value"] for h in headers if h["name"].lower() == "message-id"), None
    )

    message = MIMEText(reply_text)
    message["to"] = from_email
    message["subject"] = "Re: " + subject
    if original_msg_id:
        message["In-Reply-To"] = original_msg_id
        message["References"] = original_msg_id

    raw = base64.urlsafe_b64encode(
        message.as_bytes()
    ).decode()

    sent = service.users().messages().send(
        userId="me",
        body={"raw": raw, "threadId": original["threadId"]},
    ).execute()

    # Fetch the sent message to get its Message-ID
    try:
        sent_detail = service.users().messages().get(
            userId="me", id=sent["id"], format="metadata"
        ).execute()
        sent_headers = sent_detail["payload"]["headers"]
        reply_msg_id = next(
            (h["value"] for h in sent_headers if h["name"].lower() == "message-id"),
            None
        )
    except Exception:
        reply_msg_id = None

    return {
        "gmail_id": sent["id"],
        "thread_id": sent.get("threadId"),
        "message_id": reply_msg_id,
    }

In [140]:
def mark_as_important(service, gmail_id):
    service.users().messages().modify(
        userId="me",
        id=gmail_id,
        body={"addLabelIds": ["IMPORTANT"]},
    ).execute()

def mark_as_star(service, gmail_id):
    service.users().messages().modify(
        userId="me",
        id=gmail_id,
        body={"addLabelIds": ["STARRED"]},
    ).execute()

def mark_spam(service, gmail_id):
    service.users().messages().modify(
        userId="me",
        id=gmail_id,
        body={"addLabelIds": ["SPAM"]},
    ).execute()

In [141]:
def unmark_as_important(service, gmail_id):
    service.users().messages().modify(
        userId="me",
        id=gmail_id,
        body={"removeLabelIds": ["IMPORTANT"]},
    ).execute()

def unstar_email(service, gmail_id):
    service.users().messages().modify(
        userId="me",
        id=gmail_id,
        body={"removeLabelIds": ["STARRED"]},
    ).execute()

def unspam_email(service, gmail_id):
    service.users().messages().modify(
        userId="me",
        id=gmail_id,
        body={"removeLabelIds": ["SPAM"]},
    ).execute()

In [142]:
service = authenticate()

Credentials loaded from token.json.


In [143]:
mails = read_emails(service)
mails

[{'id': '19ca04238c91ccbc',
  'threadId': '19ca02f1dcf1e46a',
  'labelIds': ['SENT'],
  'snippet': 'Thats great, looking forward to it!',
  'internalDate': '1772215220000',
  'sizeEstimate': 804,
  'to': 'Sai Varun Chowdary Poludasu <saivarunchowdarypoludasu4248@gmail.com>',
  'subject': 'Re: Re: Lets meet',
  'in-reply-to': '<CAOFPPOHyH15wa7bw3v57FO4Yj_Sw3b9tv7=yM+tQ7KyHpbXUig@mail.gmail.com>',
  'references': '<CAOFPPOHyH15wa7bw3v57FO4Yj_Sw3b9tv7=yM+tQ7KyHpbXUig@mail.gmail.com>',
  'date': 'Fri, 27 Feb 2026 10:00:20 -0800',
  'message-id': '<CAGH8zUS0ZaDoyN=9G1hoC8n-2w2oevZZ1=RGW1hHYJUvYte_dg@mail.gmail.com>',
  'from': 'apexconnect40@gmail.com',
  'body_text': 'Thats great, looking forward to it!'},
 {'id': '19ca0403a85ad4a4',
  'threadId': '19ca02f1dcf1e46a',
  'labelIds': ['UNREAD', 'IMPORTANT', 'CATEGORY_PERSONAL', 'INBOX'],
  'snippet': 'How about tomorrow afternoon? On Fri, 27 Feb 2026 at 23:26, &lt;apexconnect40@gmail.com&gt; wrote: Sure, lets catch up soon!',
  'internalDate'

In [144]:
mark_as_important(service, "19ca02f1dcf1e46a")

In [145]:
mark_as_star(service, "19ca02f1dcf1e46a")

In [146]:
# new_reply = reply_email(service,"19ca0403a85ad4a4","Thats great, looking forward to it!")